In [ ]:
from __future__ import annotations
from typing import List

import re
import h5py
import numpy as np
from pathlib import Path

import astropy.units as u

import logging
logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import matplotlib.font_manager as fm
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar

import seaborn as sns

# Facecolor values (#f4f0e8) from S. Conradi @S_Conradi/@profConradi
custom_settings = {
    'figure.facecolor': '#ffffff',
    'axes.facecolor': '#ffffff',
    'axes.edgecolor': '0.3',
    'axes.linewidth' : '0.5',
    'axes.grid': False,
    'grid.color': '0.7',
    'grid.linestyle': ':',
    'grid.alpha': 0.6,
    'xtick.bottom': True,
    'xtick.top': True,
    'ytick.left': True,
    'ytick.right': True,
}
for t in ['xtick', 'ytick']:
    custom_settings[f'{t}.direction'] = 'in'
    custom_settings[f'{t}.color'] = '0.3'
    for m in ['major', 'minor']:
        custom_settings[f'{t}.{m}.width'] = 0.5
        custom_settings[f'{t}.{m}.size'] = 6 if m == 'major' else 3
sns.set_theme(palette=sns.color_palette('deep', as_cmap=False),
              rc=custom_settings)

In [ ]:
def add_sizebar(ax, size, fontsize):
    fontprops = fm.FontProperties(size=fontsize)
    return AnchoredSizeBar(
        ax.transData,
        size=size,
        label=f'{size:.0f} kpc',
        loc='upper right',
        pad=0.4,              # gap between bar and axes frame
        borderpad=0.5,        # gap between bar and label
        sep=4,                # pixels between bar and text
        frameon=False,        # drop the white box around it
        size_vertical=0.04,   # thickness of the bar (in axes‐fraction units)
        fontproperties=fontprops
    )

In [ ]:
class UnsupportedFormatError(RuntimeError):
    '''Raised when CosmoIO encounters an unknown file format.'''


class CosmoIO:
    '''
    A stateless class for loading cosmological snapshots from various
    formats (ASCII, HDF5, NPY, Gadget) and returning them as a
    structured numpy array.
    '''
    @staticmethod
    def load_snapshot(path: Path, *args, **kwargs):
        '''
        Loads a Gadget-format snapshot of a cosmological simulation from
        either an ASCII, HDF5, NPY or Gadget-format input file.

        Parameters
        ----------
        fname : str
            Path to the *first* snapshot file on disk. All accompanying
            parts (e.g. ``snap_001``, ``snap_002``, etc.) are detected
            and concatenated automatically for Gadget or HDF5 inputs.
        part_type : int
            The particle type to load. For Gadget snapshots, this is the
            part type index (1 for gas, 2 for dark matter, etc.). For HDF5
            snapshots, this is the part type group (e.g. 'PartType1').
        constant_res : bool
            If True, the snapshot is assumed to have constant mass resolution.
        dtype : numpy.dtype
            The data type to use for the snapshot. If not specified, float32 is used.
        '''
        path = path.expanduser().resolve()
        if not path.parent.exists():
            raise FileNotFoundError(path.parent)

        ext = CosmoIO._get_extension(path)
        loader = CosmoIO._find_loader(ext)
        files = CosmoIO._collect_files(path)
        if not files:
            raise FileNotFoundError(f'No files found matching {path.name}.')
        return loader(files, *args, **kwargs)

    @staticmethod
    def _match_extension(path: Path):
        '''
        Detects the file extension of the given path.

        Parameters
        ----------
        path : pathlib.Path
            The path to the file whose extension is to be detected.

        Returns
        -------
        match : re.Match or None
            A match object if the path matches a valid pattern, or `None` if
            it does not.
        '''
        parser_re = re.compile(
            r"^(?P<stem>.*?)"
            r"(?:"
            # Pattern 1: .<index>.<ext> (e.g. ".0.hdf5")
            r"\.(?P<index1>\d+)\.(?P<ext1>[a-zA-Z_][a-zA-Z0-9_]*)"
            r"|"
            # Pattern 2: .<ext>.<index> (e.g. ".hdf5.0")
            r"\.(?P<ext2>[a-zA-Z_][a-zA-Z0-9_]*)\.(?P<index2>\d+)"
            r"|"
            # Pattern 3: .<ext> (e.g. ".hdf5")
            r"\.(?P<ext3>[a-zA-Z_][a-zA-Z0-9_]*)"
            r")$"
        )
        return parser_re.match(path.name)

    @staticmethod
    def _get_extension(path: Path):
        '''
        Returns the file extension of the given path without the leading dot.

        Parameters
        ----------
        path : pathlib.Path
            The path to the file whose extension is to be returned.

        Returns
        -------
        ext : str or None
            The file extension without the leading dot, or `None` if no
            valid extension is found.
        '''
        match = CosmoIO._match_extension(path)
        if match:
            parts = match.groupdict()
            return re.escape(parts['ext1'] or parts['ext2'] or parts['ext3'])
        return None

    @staticmethod
    def _collect_files(path: Path):
        '''
        Gathers all snapshot files belonging to the same group.

        Parameters
        ----------
        path : pathlib.Path
            The path to any single file in the snapshot set.

        Returns
        -------
        files : List[pathlib.Path]
            A sorted list of Path objects for all files in the snapshot.
            Returns an empty list if the filepath does not match a valid
            pattern.
        '''
        match = CosmoIO._match_extension(path)

        if match:
            # Regex looking for: \.digits\.ext OR \.ext\.digits OR \.ext
            parts = match.groupdict()
            stem = re.escape(parts['stem'])
            ext = re.escape(parts['ext1'] or parts['ext2'] or parts['ext3'])
            search_pattern = rf'^{stem}(\.\d+\.{ext}|\.{ext}\.\d+|\.{ext})$'
        else:
            # If no extension is provided, treat it as a gadget file
            ext_pattern = re.compile(r"^(?P<stem>.*?)(?:\.(?P<index>\d+))?$")
            match = ext_pattern.match(path.name)
            if not match:
                return [path.name]  # Standalone gadget snapshot
            search_pattern = rf'^{re.escape(match.group("stem"))}(?:\.\d+)?$'
        search_re = re.compile(search_pattern)

        files = []
        for f in path.parent.iterdir():
            if f.is_file() and search_re.match(f.name):
                files.append(f)
        return sorted(files)


    @staticmethod
    def _find_loader(ext: str):
        '''Return the appropriate backend reader for ``path``.'''
        _LOADER_MAP = {
            "dat": lambda f, *args, **kw: CosmoIO._load_ascii(f, **kw),
            "txt": lambda f, *args, **kw: CosmoIO._load_ascii(f, **kw),
            "hdf5": lambda f, *args, **kw: CosmoIO._load_hdf5(f, *args, **kw),
            "h5": lambda f, *args, **kw: CosmoIO._load_hdf5(f, *args, **kw)
        }

        if ext in _LOADER_MAP:
            return _LOADER_MAP[ext]

    @staticmethod
    def _load_ascii(files: List[Path], **kwargs):
        '''Load a cosmological snapshot from an ASCII file.'''
        dtype = kwargs.get('dtype', np.float32)
        particleIDs, coordinates, velocities, masses = [], [], [], []
        logger.info(f'Reading the input ASCII files ...')
        for path in files:
            logger.info(f'Opening ASCII file {path}...')
            data = np.loadtxt(path)
            particleIDs.append(np.array(data[:, 0], dtype=np.uint64))
            coordinates.append(np.array(data[:, 1:4], dtype=dtype))
            velocities.append(np.array(data[:, 4:7], dtype=dtype))
            masses.append(np.array(data[:, 7], dtype=dtype))
        particleIDs = np.concatenate(particleIDs, dtype=np.uint64)
        coordinates = np.concatenate(coordinates, dtype=dtype)
        velocities = np.concatenate(velocities, dtype=dtype)
        masses = np.concatenate(masses, dtype=dtype)
        return particleIDs, coordinates, velocities, masses

    @staticmethod
    def _load_hdf5(files: List[Path], *args, **kwargs):
        '''Load a cosmological snapshot from an HDF5 file.'''
        logger.info(f'Reading the input HDF5 files ...')
        part_type = kwargs.get('part_type', 1)
        if args:
            arguments = {ai: [] for ai in args}
            dtypes = {ai: None for ai in args}
        for f in files:
            logger.info(f'Opening HDF file {f}...')
            with h5py.File(f, 'r') as hdf:
                for ai in args:
                    arguments[ai].append(hdf[f'/PartType{part_type}/{ai}'][:])
                    dtypes[ai] = hdf[f'/PartType{part_type}/{ai}'].dtype
                if 'Masses' in args:
                    N_part = hdf['/Header'].attrs['NumPart_ThisFile'][part_type]
                    mass_part_type = hdf['/Header'].attrs['MassTable'][part_type]
                    if np.all(arguments['Masses'] == 0):
                        arguments['Masses'] = np.ones(N_part) * mass_part_type
                    if kwargs.get('constant_res', False):
                        arguments['Masses'] *= mass_part_type
        for ai in args:
            arguments[ai] = np.concatenate(arguments[ai], dtype=dtypes[ai])
        return arguments.values()

In [ ]:
path = Path('/home/pal.balazs/data/fire/snapshot_600.hdf5')

pos_gas, vel_gas, mass_gas, eab_gas, u_gas = CosmoIO.load_snapshot(path,
    'Coordinates', 'Velocities', 'Masses', 'ElectronAbundance', 'InternalEnergy',
    part_type=0)
pos_dm, vel_dm, mass_dm = CosmoIO.load_snapshot(path,
    'Coordinates', 'Velocities', 'Masses',
    part_type=1)
pos_star, vel_star, mass_star = CosmoIO.load_snapshot(path,
    'Coordinates', 'Velocities', 'Masses',
    part_type=4)

In [ ]:
part_type = 0
with h5py.File(Path('/home/pal.balazs/data/fire/snapshot_600.0.hdf5'), 'r') as hdf:
    Lbox = hdf['Header'].attrs['BoxSize']
    h = hdf['Header'].attrs['HubbleParam']
    a = hdf['Header'].attrs['Time']
    print(hdf['Header'].attrs['NumPart_ThisFile'][part_type])

In [ ]:
from scipy.constants import Boltzmann as kB
from scipy.constants import m_p

gamma = 5/3
XH, XHe = 0.76, 0.24  # H/He mass ratio
mu = 1/(XH + XHe/4 + eab_gas)  # Mean molecular weight
# possible fallback: mu = 0.59  # for fully ionised gas

T = (gamma - 1) * mu * m_p / (kB / 1e6) * (u_gas * (a**2))  # [K]

In [ ]:
def move_to_coi(x, coi):
    return x - np.broadcast_to(coi, (3,))
def mask_sphere(x, R=20):
    return np.linalg.norm(x, axis=1) < R
def move_to_com(x, m):
    com = np.sum(x * m[:, None], axis=0) / np.sum(m)
    return move_to_coi(x, coi=com)
def centre_of_potential(x, m, r_init=20, n_iter=3):
    cop = x[np.argmin(np.linalg.norm(x, axis=1))]
    for _ in range(n_iter):
        d = np.linalg.norm(x - cop, axis=1)
        inner = d < r_init
        cop = np.average(x[inner], axis=0, weights=m[inner])
        r_init *= 0.5
    return cop

In [ ]:
# FIRE-2 m12i-R7100
coi = [29338, 30980, 32480]
R = 22.5
h = 0.702

# Move target of interest approximately near to the center
x_gas = move_to_coi(pos_gas, coi=coi) / h
x_dm = move_to_coi(pos_dm, coi=coi) / h
x_star = move_to_coi(pos_star, coi=coi) / h

# Cut a sphere from around the center
mask_gas = mask_sphere(x_gas, R)
x_gas, v_gas = x_gas[mask_gas], vel_gas[mask_gas]
mask_dm = mask_sphere(x_dm, R)
x_dm, v_dm = x_dm[mask_dm], vel_dm[mask_dm]
mask_star = mask_sphere(x_star, R)
x_star, v_star = x_star[mask_star], vel_star[mask_star]
# Rescale masses to true values
m_gas = mass_gas[mask_gas]*1e10 / h
T_gas = T[mask_gas]
m_dm = mass_dm[mask_dm]*1e10 / h
m_star = mass_star[mask_star]*1e10 / h

# find COM/CoP of gas disk
cop = centre_of_potential(x_gas, m_gas, r_init=R, n_iter=5)
v_bulk = np.average(v_gas, axis=0, weights=m_gas)

# shift *everything*
x_gas_ua = x_gas - cop
v_gas_ua = v_gas - v_bulk
x_dm_ua = x_dm - cop
v_dm_ua = v_dm - v_bulk
x_star_ua = x_star - cop
v_star_ua = v_star - v_bulk

# rotate galaxy in plane
pcs_gas = np.linalg.svd(x_gas_ua - x_gas_ua.mean(0), full_matrices=False)[2]
x_gas = (x_gas_ua - x_gas_ua.mean(0)) @ pcs_gas.T
v_gas = (v_gas_ua - v_gas_ua.mean(0)) @ pcs_gas.T
x_dm = (x_dm_ua - x_dm_ua.mean(0)) @ pcs_gas.T
v_dm = (v_dm_ua - v_dm_ua.mean(0)) @ pcs_gas.T
x_star = (x_star_ua - x_star_ua.mean(0)) @ pcs_gas.T
v_star = (v_star_ua - v_star_ua.mean(0)) @ pcs_gas.T

In [ ]:
nr, nc = 1, 3
fig, axes = plt.subplots(nr, nc, figsize=(nc*4, nr*4), dpi=400)

labels = ['x', 'y', 'z']
for i, ax in enumerate(axes.flat[:3]):
    ax.set_aspect(1)
    idx = [k for k in range(3) if k != i]

    ax.scatter(*x_gas_ua[::10, [idx[0], idx[1]]].T,
               color='black', s=0.01, alpha=0.1, rasterized=True)
    for pci in pcs_gas:
        ax.plot([0, pci[idx[0]]*10], [0, pci[idx[1]]*10], lw=2)

    ax.set_xlabel(f'{labels[idx[0]]} [kpc]')
    ax.set_ylabel(f'{labels[idx[1]]} [kpc]')
    ax.set_xlim(-R, R)
    ax.set_ylim(-R, R)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.add_artist(add_sizebar(ax, size=5, fontsize=6))

#fig.savefig('galaxies-pca.pdf', bbox_inches='tight', pad_inches=0.1)
plt.show()

In [ ]:
nr, nc = 1, 3
fig, axes = plt.subplots(nr, nc, figsize=(nc*4, nr*4), dpi=400)

labels = ['x', 'y', 'z']
for i, ax in enumerate(axes.flat[:3]):
    ax.set_aspect(1)
    idx = [k for k in range(3) if k != i]

    ax.scatter(*x_gas[:, [idx[0], idx[1]]].T,
               color='black', ec='none', s=0.1, alpha=0.1, rasterized=True)

    ax.set_xlabel(f'{labels[idx[0]]} [kpc]')
    ax.set_ylabel(f'{labels[idx[1]]} [kpc]')
    ax.set_xlim(-R, R)
    ax.set_ylim(-R, R)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.add_artist(add_sizebar(ax, size=5, fontsize=6))

#fig.savefig('galaxies-gas-rotated.pdf', bbox_inches='tight', pad_inches=0.1)
plt.show()

In [ ]:
nr, nc = 1, 3
fig, axes = plt.subplots(nr, nc, figsize=(nc*4, nr*4), dpi=400)

labels = ['x', 'y', 'z']
for i, ax in enumerate(axes.flat[:3]):
    ax.set_aspect(1)
    idx = [k for k in range(3) if k != i]

    ax.scatter(*x_star[::10, [idx[0], idx[1]]].T,
               color='black', s=0.01, alpha=0.1, rasterized=True)

    ax.set_xlabel(f'{labels[idx[0]]} [kpc]')
    ax.set_ylabel(f'{labels[idx[1]]} [kpc]')
    ax.set_xlim(-R, R)
    ax.set_ylim(-R, R)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.add_artist(add_sizebar(ax, size=5, fontsize=6))

#fig.savefig('galaxies-star-rotated.pdf', bbox_inches='tight', pad_inches=0.1)
plt.show()

In [ ]:
def mask_cone(x, k, phi=5):
    k = k / np.linalg.norm(k)
    xn = x / np.linalg.norm(x, axis=1)[:, None]
    cos_phi = np.cos(np.deg2rad(phi) / 2)
    return np.dot(xn, k) > cos_phi
def mask_belt(x, k, phi=5):
    k = k / np.linalg.norm(k)
    xn = x / np.linalg.norm(x, axis=1)[:, None]
    sin_phi = np.sin(np.deg2rad(phi) / 2)
    return np.abs(np.dot(xn, k)) < sin_phi
def v_tangential(v, k):
    k = k / np.linalg.norm(k)
    return np.cross(k, v)

In [ ]:
from scipy.constants import G as G_SI  # m^3 kg^-1 s^-2
G = G_SI * (u.m*u.m*u.m / u.kg / u.s/u.s)
G = G.to(u.kpc*u.km*u.km / u.s/u.s / u.Msun)
G = G.value  # kpc (km/s)^2 / M_sun

# velocity bins
nbin = 40
bins = np.linspace(0, R, nbin + 1)
centers = 0.5*(bins[1:] + bins[:-1])

# Arrays for summary statistics
v_med = np.full_like(centers, np.nan)
v_p16 = v_med.copy(); v_p84 = v_med.copy()
v_p025 = v_med.copy(); v_p975 = v_med.copy()

# constituent distance from center
r_gas = np.linalg.norm(x_gas, axis=1)
r_dm = np.linalg.norm(x_dm, axis=1)
r_star = np.linalg.norm(x_star, axis=1)

# combine baryonic matter (star + gas)
# gas: keep only cold (T < 2e4 K) and rotationally supported
#jz_jcirc_gas = ...
mask_gas = (T_gas < 2e4)# & (jz_jcirc_gas > 0.7)
# stars: rotationally supported and near the plane (|z| < 2 kpc)
#jz_jcirc_star = ...
mask_star = (np.abs(x_star[:, 2]) < 2.0)# & (jz_jcirc_star > 0.7)

x_bary = np.vstack((x_gas[mask_gas], x_star[mask_star]))
v_bary = np.vstack((v_gas[mask_gas], v_star[mask_star]))
m_bary = np.concatenate((m_gas[mask_gas], m_star[mask_star]))
r_bary = np.linalg.norm(x_bary, axis=1)

# pre–compute distances and tangential speeds
v_t = np.linalg.norm(v_tangential(v=v_bary, k=[0, 0, 1]), axis=1)
for i in range(nbin):
    in_bin = (r_bary >= bins[i]) & (r_bary < bins[i+1])
    if not np.any(in_bin):
        continue
    v_bin = v_t[in_bin]
    v_med[i] = np.median(v_bin)
    v_p16[i], v_p84[i] = np.percentile(v_bin, [16, 84])
    v_p025[i], v_p975[i] = np.percentile(v_bin, [2.5, 97.5])

# Keplerian curve from baryons only
v_kep_bary = v_med.copy()
for i, r_c in enumerate(centers):
    m_enc_bary = m_bary[r_bary < r_c].sum()
    v_kep_bary[i] = np.sqrt(G*m_enc_bary / r_c)

# Dark-matter only
v_kep_dm = v_med.copy()
for i, r_c in enumerate(centers):
    m_enc_dm = m_dm[r_dm < r_c].sum()
    v_kep_dm[i] = np.sqrt(G*m_enc_dm / r_c)

# Combine gas + star + DM
v_total = np.sqrt(v_kep_bary**2 + v_kep_dm**2)

In [ ]:
print(f"Total mass inside {R} kpc : {m_bary.sum():.2e} Msun")
print(f"v_Kep( 5 kpc) ~ {np.sqrt(G * m_bary[r_bary<5].sum()/5):.1f} km/s")
print(f"v_Kep(10 kpc) ~ {np.sqrt(G * m_bary[r_bary<10].sum()/10):.1f} km/s")
print(f"v_Kep(20 kpc) ~ {np.sqrt(G * m_bary[r_bary<20].sum()/20):.1f} km/s")

In [ ]:
# thank you chatgpt
with mpl.rc_context({
    "backend": "pgf",
    "pgf.texsystem": "pdflatex",
    "text.usetex": True,               # route *all* text through LaTeX
    "font.family": "serif",            # rely on document’s \rmfamily
    "pgf.rcfonts": False,
    "font.size": 10
}):
    nr, nc = 1, 2
    fig, axes = plt.subplots(nr, nc, figsize=(nc*4.5, nr*4), dpi=400)

    ax = axes[0]
    ax.set_box_aspect(1)
    ax.scatter(*x_gas[:, [0, 1]].T,
               color='black', ec='none', s=0.1, alpha=0.1, rasterized=True)
    ax.set_xlabel('PC$_{1}$ [kpc]', fontsize=12)
    ax.set_ylabel('PC$_{2}$ [kpc]', fontsize=12)
    ax.set_xlim(-R, R)
    ax.set_ylim(-R, R)
    ax.add_artist(add_sizebar(ax, size=5, fontsize=8))

    ax = axes[1]
    ax.set_box_aspect(1)
    #ax.scatter(np.linalg.norm(x_gas, axis=1),
    #           np.linalg.norm(v_gas_tan, axis=1),
    #          color='black', ec='none', s=0.01, alpha=1.0, rasterized=True)
    ax.plot(centers, v_med, color='crimson', lw=1.5, label='median')
    ax.fill_between(centers, v_p16,  v_p84,  color='crimson', alpha=0.25,
                    label=r'$68~\%$ range')
    ax.fill_between(centers, v_p025, v_p975, color='crimson', alpha=0.12,
                    label=r'$95~\%$ range')

    ax.plot(centers, v_kep_bary, ls='-.', color='grey', lw=1.5,
            label=r'$v_{\rm Kep}(r)$')

    ax.set_xlim(0, R)
    ax.set_ylim(0, 500)
    ax.set_xlabel(r'$r_{\mathrm{center}}$ [kpc]', fontsize=12)
    ax.set_ylabel(r'$v_{\mathrm{circ}}$ [km s$^{-1}$]', fontsize=12)
    #ax.set_title('Tangential velocity [km/s]', fontsize=10, loc='left')
    #handles = [Line2D([0], [0], label='Tangential velocity [km/s]',
    #                  color='black', marker='none', lw=2)]
    ax.legend(fontsize=8, frameon=False, loc='upper right')

    for ax in axes:
        ax.tick_params(axis='both', which='major', labelsize=12)

    fig.savefig('rotation-curve.pdf', bbox_inches='tight', pad_inches=0.1)
    plt.close()